In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../outputs/merged_data.csv')

nlp_df = df[['TransactionID', 'isFraud', 'P_emaildomain', 'R_emaildomain', 'DeviceInfo']].copy()

print("Shape:", nlp_df.shape)
print("\nNull counts:")
print(nlp_df.isnull().sum())

Shape: (590540, 5)

Null counts:
TransactionID         0
isFraud               0
P_emaildomain     94456
R_emaildomain    453249
DeviceInfo       471874
dtype: int64


In [2]:
print("Unique P_emaildomain values:", nlp_df['P_emaildomain'].nunique())
print("Unique R_emaildomain values:", nlp_df['R_emaildomain'].nunique())
print("Unique DeviceInfo values:", nlp_df['DeviceInfo'].nunique())

Unique P_emaildomain values: 59
Unique R_emaildomain values: 60
Unique DeviceInfo values: 1786


In [3]:
# Fill nulls
nlp_df['P_emaildomain'] = nlp_df['P_emaildomain'].fillna('unknown')
nlp_df['R_emaildomain'] = nlp_df['R_emaildomain'].fillna('unknown')

# Extract TLD (top level domain)
nlp_df['P_email_tld'] = nlp_df['P_emaildomain'].apply(lambda x: x.split('.')[-1] if x != 'unknown' else 'unknown')

# Domain type — free, corporate, anonymous, unknown
def classify_domain(domain):
    free = ['gmail.com', 'yahoo.com', 'hotmail.com', 'aol.com', 'icloud.com',
            'outlook.com', 'msn.com', 'live.com', 'ymail.com', 'me.com']
    anonymous = ['anonymous.com', 'mail.com', 'protonmail.com']
    if domain == 'unknown':
        return 'unknown'
    elif domain in anonymous:
        return 'anonymous'
    elif domain in free:
        return 'free'
    else:
        return 'corporate'

nlp_df['P_email_type'] = nlp_df['P_emaildomain'].apply(classify_domain)

print("P_email_type distribution:")
print(nlp_df['P_email_type'].value_counts())

P_email_type distribution:
P_email_type
free         425242
unknown       94456
anonymous     37633
corporate     33209
Name: count, dtype: int64


In [4]:
# Fraud rate per P_emaildomain
p_email_fraud_rate = nlp_df.groupby('P_emaildomain')['isFraud'].mean()
nlp_df['P_email_fraud_rate'] = nlp_df['P_emaildomain'].map(p_email_fraud_rate)

# Fraud rate per email type
type_fraud_rate = nlp_df.groupby('P_email_type')['isFraud'].mean()
nlp_df['P_email_type_fraud_rate'] = nlp_df['P_email_type'].map(type_fraud_rate)

# Domain mismatch flag — P and R email domains differ (where R exists)
nlp_df['R_emaildomain'] = nlp_df['R_emaildomain'].fillna('unknown')
nlp_df['email_domain_mismatch'] = (
    nlp_df['P_emaildomain'] != nlp_df['R_emaildomain']
).astype(int)

print("Email fraud rates by type:")
print(type_fraud_rate)
print("\nDomain mismatch rate:", round(nlp_df['email_domain_mismatch'].mean() * 100, 2), "%")

Email fraud rates by type:
P_email_type
anonymous    0.026466
corporate    0.020898
free         0.038056
unknown      0.029538
Name: isFraud, dtype: float64

Domain mismatch rate: 68.52 %


In [5]:
def parse_device(device):
    if pd.isna(device) or device == 'unknown':
        return 'unknown'
    device = str(device).lower()
    if 'windows' in device:
        return 'windows'
    elif 'ios' in device or 'iphone' in device or 'ipad' in device:
        return 'ios'
    elif 'macos' in device or 'mac os' in device:
        return 'macos'
    elif 'sm-' in device or 'samsung' in device:
        return 'samsung_android'
    elif 'huawei' in device or 'ale-' in device or 'hol-' in device:
        return 'huawei_android'
    elif 'trident' in device:
        return 'ie_browser'
    elif 'rv:' in device:
        return 'firefox_browser'
    elif 'android' in device:
        return 'other_android'
    else:
        return 'other'

nlp_df['device_family'] = nlp_df['DeviceInfo'].apply(parse_device)

print("Device family distribution:")
print(nlp_df['device_family'].value_counts())

Device family distribution:
device_family
unknown            471874
windows             47775
ios                 19783
macos               12573
other               12273
samsung_android     11940
ie_browser           7440
firefox_browser      4385
huawei_android       2415
other_android          82
Name: count, dtype: int64


In [6]:
device_fraud_rate = nlp_df.groupby('device_family')['isFraud'].mean()
nlp_df['device_fraud_rate'] = nlp_df['device_family'].map(device_fraud_rate)

print("Fraud rate by device family:")
print(device_fraud_rate.sort_values(ascending=False))

Fraud rate by device family:
device_family
other              0.148212
huawei_android     0.135818
samsung_android    0.115410
firefox_browser    0.075941
windows            0.065536
ios                0.062731
other_android      0.036585
unknown            0.025549
macos              0.022111
ie_browser         0.012903
Name: isFraud, dtype: float64


In [7]:
nlp_features = nlp_df[[
    'TransactionID',
    'P_email_fraud_rate',
    'P_email_type_fraud_rate',
    'email_domain_mismatch',
    'device_fraud_rate'
]].copy()

# Label encode device family and email type for model use
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
nlp_features['P_email_type_encoded'] = le.fit_transform(nlp_df['P_email_type'])
nlp_features['device_family_encoded'] = le.fit_transform(nlp_df['device_family'])

print("NLP features shape:", nlp_features.shape)
print("\nSample:")
print(nlp_features.head())

nlp_features.to_csv('../outputs/nlp_features.csv', index=False)
print("\nSaved to outputs/nlp_features.csv")

NLP features shape: (590540, 7)

Sample:
   TransactionID  P_email_fraud_rate  P_email_type_fraud_rate  \
0        2987000            0.029538                 0.029538   
1        2987001            0.043542                 0.038056   
2        2987002            0.094584                 0.038056   
3        2987003            0.022757                 0.038056   
4        2987004            0.043542                 0.038056   

   email_domain_mismatch  device_fraud_rate  P_email_type_encoded  \
0                      0           0.025549                     3   
1                      1           0.025549                     2   
2                      1           0.025549                     2   
3                      1           0.025549                     2   
4                      1           0.115410                     2   

   device_family_encoded  
0                      8  
1                      8  
2                      8  
3                      8  
4                 

## NLP Module Results

Text columns processed: `P_emaildomain`, `R_emaildomain`, `DeviceInfo`

**Email domain features engineered:**
- `P_email_fraud_rate` — target-encoded fraud rate per domain (mail.com = 19%, outlook = 9.4%)
- `P_email_type_fraud_rate` — fraud rate by domain category (free providers highest at 3.8%)
- `email_domain_mismatch` — flag where purchaser and recipient domains differ
- `P_email_type_encoded` — domain category label encoded (free/corporate/anonymous/unknown)

**Device features engineered:**
- `device_fraud_rate` — target-encoded fraud rate per device family
- `device_family_encoded` — parsed device family label encoded
- Key finding: Huawei (13.6%) and Samsung Android (11.5%) have 3-4x average fraud rate
- macOS (2.2%) and IE browser (1.3%) are the lowest risk device families

**Key insight:** Free email providers (gmail, yahoo, hotmail) have HIGHER fraud rates than 
anonymous domains — fraudsters use real providers, not just throwaway emails.